# Geovisor - Hospitales y Clinicas ITT

**Proyecto:** Gobierno de Datos 2026 - Infraestructura de Salud Cali

**Modulo:** Espacializacion de Instituciones Prestadoras de Salud

**Repositorio:** [GitHub](https://github.com/j0rg3c45/Hospitales_Cl-nicas_ITT.git)

---

## Objetivo

Generar un visor geografico interactivo y un analisis descriptivo de los 43 centros medicos del proyecto de infraestructura hospitalaria de Cali.

---
## 1. Instalacion de Dependencias

In [ ]:
%pip install folium matplotlib --quiet

---
## 2. Importacion de Librerias

In [ ]:
import json
import folium
from folium import FeatureGroup, LayerControl, GeoJson
from folium.plugins import MiniMap, Fullscreen, LocateControl
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from collections import Counter

print("Librerias importadas.")

---
## 3. Carga de Datos

In [ ]:
# Clonar repositorio para acceder a los datos
import os
if not os.path.exists('Hospitales_Cl-nicas_ITT'):
    !git clone https://github.com/j0rg3c45/Hospitales_Cl-nicas_ITT.git

DATA_PATH = 'Hospitales_Cl-nicas_ITT/Data/centros_medicos_completo.json'

if not os.path.exists(DATA_PATH):
    from google.colab import files
    print('Suba el archivo centros_medicos_completo.json:')
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    centros = json.load(f)

print(f'Centros medicos cargados: {len(centros)}')

---
## 4. Tabla Resumen

In [ ]:
# Tabla resumen de todos los centros
header = f"{'#':<3} {'Nombre':<40} {'ESE':<18} {'Comuna':<12} {'Estado':<15} {'Usuarios':>10} {'Monto':>15}"
print(header)
print('-' * len(header))

for i, c in enumerate(centros):
    monto_str = f"${c['monto']:,.0f}" if c['monto'] > 0 else 'Pendiente'
    print(f"{i+1:<3} {c['nombre'][:39]:<40} {c['ese'][:17]:<18} {c['comuna'][:11]:<12} {c['estado'][:14]:<15} {c['usuarios']:>10,} {monto_str:>15}")

---
## 5. Analisis Grafico

In [ ]:
# Configuracion de graficos
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Analisis de Infraestructura Hospitalaria - Cali 2026', fontsize=14, fontweight='bold')

# --- Grafico 1: Distribucion por ESE ---
eses = Counter(c['ese'] for c in centros)
ax1 = axes[0, 0]
bars = ax1.barh(list(eses.keys()), list(eses.values()), color=['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336'])
ax1.set_title('Centros por ESE')
ax1.set_xlabel('Cantidad')
for bar in bars:
    ax1.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2, f'{int(bar.get_width())}', va='center')

# --- Grafico 2: Distribucion por Estado ---
estados = Counter(c['estado'] for c in centros)
ax2 = axes[0, 1]
colores_estado = {'Finalizado': '#4CAF50', 'En ejecucion': '#FF9800', 'Prefactibilidad': '#9E9E9E',
                  'En revision para concepto de viabilidad': '#2196F3',
                  'Necesidad manifestada por la ese para la vigencia 2026': '#F44336',
                  'En alistamiento': '#795548'}
labels_estado = [k[:25] for k in estados.keys()]
colors_pie = [colores_estado.get(k, '#9E9E9E') for k in estados.keys()]
ax2.pie(estados.values(), labels=labels_estado, autopct='%1.0f%%', colors=colors_pie, startangle=90)
ax2.set_title('Estado de los Proyectos')

# --- Grafico 3: Top 10 por Usuarios ---
top_usuarios = sorted(centros, key=lambda x: x['usuarios'], reverse=True)[:10]
ax3 = axes[1, 0]
nombres_cortos = [c['nombre'][:25] for c in top_usuarios]
valores_usuarios = [c['usuarios'] for c in top_usuarios]
ax3.barh(nombres_cortos[::-1], valores_usuarios[::-1], color='#2196F3')
ax3.set_title('Top 10 - Mayor Poblacion Beneficiaria')
ax3.set_xlabel('Usuarios')

# --- Grafico 4: Top 10 por Monto de Inversion ---
top_monto = sorted([c for c in centros if c['monto'] > 0], key=lambda x: x['monto'], reverse=True)[:10]
ax4 = axes[1, 1]
nombres_monto = [c['nombre'][:25] for c in top_monto]
valores_monto = [c['monto'] / 1_000_000_000 for c in top_monto]  # En miles de millones
ax4.barh(nombres_monto[::-1], valores_monto[::-1], color='#4CAF50')
ax4.set_title('Top 10 - Mayor Inversion (Miles de Millones COP)')
ax4.set_xlabel('Monto (Miles de Millones)')

plt.tight_layout()
plt.savefig('analisis_infraestructura.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graficos generados.')

In [ ]:
# --- Estadisticas resumen ---
total_usuarios = sum(c['usuarios'] for c in centros)
total_inversion = sum(c['monto'] for c in centros if c['monto'] > 0)
finalizados = sum(1 for c in centros if c['estado'] == 'Finalizado')
en_ejecucion = sum(1 for c in centros if 'ejecuci' in c['estado'].lower())

print('RESUMEN GENERAL')
print('=' * 50)
print(f'Total centros medicos:      {len(centros)}')
print(f'Finalizados:                {finalizados}')
print(f'En ejecucion:               {en_ejecucion}')
print(f'Otros estados:              {len(centros) - finalizados - en_ejecucion}')
print(f'Poblacion beneficiaria:     {total_usuarios:,}')
print(f'Inversion total:            ${total_inversion:,.0f}')
print(f'Inversion promedio/centro:  ${total_inversion//len(centros):,.0f}')

---
## 6. Espacializacion / Geovisor

Mapa interactivo con:
- Marcadores coloreados por estado del proyecto.
- Layer Control agrupado por ESE.
- Popups con toda la informacion de cada centro.
- Basemaps de Google Satellite/Hybrid/Maps.

In [ ]:
# Tiles de Google
GOOGLE_SATELLITE = 'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}'
GOOGLE_MAPS = 'https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}'
GOOGLE_HYBRID = 'https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}'

# Centro del mapa
center_lat = sum(c['lat'] for c in centros) / len(centros)
center_lon = sum(c['lon'] for c in centros) / len(centros)

# Crear mapa
mapa = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles=None,
    control_scale=True
)

# Basemaps
folium.TileLayer(tiles=GOOGLE_SATELLITE, attr='Google', name='Google Satellite', overlay=False).add_to(mapa)
folium.TileLayer(tiles=GOOGLE_HYBRID, attr='Google', name='Google Hybrid', overlay=False).add_to(mapa)
folium.TileLayer(tiles=GOOGLE_MAPS, attr='Google', name='Google Maps', overlay=False).add_to(mapa)
folium.TileLayer(tiles='OpenStreetMap', name='OpenStreetMap', overlay=False).add_to(mapa)

# Colores por estado
def color_por_estado(estado):
    if estado == 'Finalizado':
        return 'green'
    elif 'ejecuci' in estado.lower():
        return 'orange'
    elif 'revision' in estado.lower() or 'alistamiento' in estado.lower():
        return 'blue'
    elif 'Necesidad' in estado or 'Prefactibilidad' in estado:
        return 'red'
    return 'gray'

# Agrupar por ESE
eses_unicas = sorted(set(c['ese'] for c in centros))

for ese in eses_unicas:
    fg = FeatureGroup(name=ese, show=True)

    centros_ese = [c for c in centros if c['ese'] == ese]

    for c in centros_ese:
        color = color_por_estado(c['estado'])
        monto_str = f"${c['monto']:,.0f}" if c['monto'] > 0 else 'Pendiente'

        popup_html = (
            f"<div style='font-family:Arial;font-size:11px;min-width:280px;max-height:350px;overflow-y:auto;'>"
            f"<h4 style='margin-bottom:5px;color:#333;'>{c['nombre']}</h4>"
            f"<hr style='margin:3px 0;'>"
            f"<b>ID:</b> {c['id']}<br>"
            f"<b>Tipo:</b> {c['tipo_ips']}<br>"
            f"<b>ESE:</b> {c['ese']}<br>"
            f"<b>Direccion:</b> {c['direccion']}<br>"
            f"<b>Barrio:</b> {c['barrio']}<br>"
            f"<b>Comuna:</b> {c['comuna']}<br>"
            f"<hr style='margin:3px 0;'>"
            f"<b>Usuarios:</b> {c['usuarios']:,}<br>"
            f"<b>Ambientes actuales:</b> {c['ambientes_actuales']}<br>"
            f"<b>Ambientes proyectados:</b> {c['ambientes_proyectados']}<br>"
            f"<b>Servicios actuales:</b> {c['servicios_actuales']}<br>"
            f"<b>Servicios nuevos:</b> {c['servicios_nuevos']}<br>"
            f"<hr style='margin:3px 0;'>"
            f"<b>Estado:</b> {c['estado']}<br>"
            f"<b>Inicio:</b> {c['fecha_inicio']}<br>"
            f"<b>Fin:</b> {c['fecha_fin']}<br>"
            f"<b>Monto:</b> {monto_str}<br>"
            f"<b>Fuente:</b> {c['fuente']}<br>"
            f"</div>"
        )

        folium.Marker(
            location=[c['lat'], c['lon']],
            popup=folium.Popup(popup_html, max_width=350),
            tooltip=f"{c['nombre']} ({c['estado']})",
            icon=folium.Icon(color=color, icon='plus-sign', prefix='glyphicon')
        ).add_to(fg)

    fg.add_to(mapa)

# Controles
LayerControl(collapsed=False, position='topright').add_to(mapa)
MiniMap(toggle_display=True, position='bottomleft').add_to(mapa)
Fullscreen(position='topleft', title='Pantalla completa', title_cancel='Salir').add_to(mapa)
LocateControl(strings={'title': 'Mi ubicacion'}).add_to(mapa)

print(f'Geovisor: {len(centros)} centros en {len(eses_unicas)} capas (ESE).')
print(f'Colores: verde=Finalizado, naranja=En ejecucion, azul=En revision, rojo=Pendiente')

---
## 7. Visualizacion del Geovisor

In [ ]:
display(mapa)

---
## 8. Exportacion

In [ ]:
OUTPUT_FILE = 'geovisor_hospitales_clinicas_ITT.html'
mapa.save(OUTPUT_FILE)
print(f'Mapa exportado: {OUTPUT_FILE}')

try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    files.download('analisis_infraestructura.png')
except:
    print('Abra los archivos desde el explorador.')